<a href="https://colab.research.google.com/github/encen-ya/ML_2026/blob/main/Praktikum4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install annoy faiss-cpu hnswlib

import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

# ===============================
# 1. Buat dataset 1 juta data 5D
# ===============================
n_data = 1_000_000   # bisa coba 100_000 dulu jika RAM terbatas
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10

# ===============================
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

# ===============================
# 3. FAISS (Flat Index)
# ===============================
print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")

# ===============================
# 4. HNSW (hnswlib)
# ===============================
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 10.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 17.4 MB/s eta 0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520037 sha256=d8fad1e6e53c106bbae212bf90e6e32bf9032277292286cd811f17840f527d5c
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679803 sha256=c5ca873ec4712f08943f0c6c5a3829bd2c878c715e89b4719c09f28f502ce7a1
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built annoy hnswlib
=== Annoy ===
Build time: 22.903577089309692 detik
Query time: 0.00023603439331054688 det

In [4]:
import pandas as pd
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

def recall(predictions, ground_truth, k):
    """Menghitung rata-rata recall@k antara prediksi dan ground truth."""
    recalls = []
    for pred, gt in zip(predictions, ground_truth):
        actual = set(gt[:k])
        predicted = set(pred[:k])
        intersection = actual.intersection(predicted)
        recalls.append(len(intersection) / k)
    return round(np.mean(recalls), 4)

def compare_three(metric, n_data=1_000_000, dim=5, k=10, nq=100, seed=1):
    rng = np.random.default_rng(seed)
    X = rng.random((n_data, dim)).astype("float32")
    Q = rng.random((nq, dim)).astype("float32")
    Xf, Qf = X.copy(), Q.copy()
    if metric == "cosine":
        faiss.normalize_L2(Xf); faiss.normalize_L2(Qf)

    gt_index = faiss.IndexFlatL2(dim) if metric == "euclidean" else faiss.IndexFlatIP(dim)
    gt_index.add(Xf)
    _, gt = gt_index.search(Qf, k)

    out = []
    # Annoy
    am = "euclidean" if metric == "euclidean" else "angular"
    a = AnnoyIndex(dim, am)
    t0 = time.perf_counter()
    for i in range(n_data): a.add_item(i, X[i])
    a.build(10); tb = time.perf_counter() - t0
    t0 = time.perf_counter(); I = [a.get_nns_by_vector(q, k) for q in Q]; tq = (time.perf_counter() - t0) / nq * 1000
    out.append({"Metric": metric, "Model": "Annoy (10 tree)", "Build (s)": round(tb, 2), "Query (ms/q)": round(tq, 4), "Recall": recall(I, gt, k)})
    # FAISS Flat
    fi = faiss.IndexFlatL2(dim) if metric == "euclidean" else faiss.IndexFlatIP(dim)
    t0 = time.perf_counter(); fi.add(Xf); tb = time.perf_counter() - t0
    t0 = time.perf_counter(); _, I = fi.search(Qf, k); tq = (time.perf_counter() - t0) / nq * 1000
    out.append({"Metric": metric, "Model": "FAISS IndexFlat", "Build (s)": round(tb, 2), "Query (ms/q)": round(tq, 4), "Recall": recall(I, gt, k)})
    # HNSW
    h = hnswlib.Index(space="l2" if metric == "euclidean" else "cosine", dim=dim)
    t0 = time.perf_counter(); h.init_index(max_elements=n_data, ef_construction=200, M=16); h.add_items(X); tb = time.perf_counter() - t0
    h.set_ef(50)
    t0 = time.perf_counter(); I, _ = h.knn_query(Q, k=k); tq = (time.perf_counter() - t0) / nq * 1000
    out.append({"Metric": metric, "Model": "HNSW (ef=50)", "Build (s)": round(tb, 2), "Query (ms/q)": round(tq, 4), "Recall": recall(I, gt, k)})
    return out

tabel_p4 = pd.DataFrame(compare_three("euclidean") + compare_three("cosine"))
tabel_p4

,Metric,Model,Build (s),Query (ms/q),Recall
0,euclidean,Annoy (10 tree),21.52,0.0591,0.953
1,euclidean,FAISS IndexFlat,0.01,5.1658,1.000
2,euclidean,HNSW (ef=50),168.11,0.0408,1.000
3,cosine,Annoy (10 tree),27.76,0.0583,0.979
4,cosine,FAISS IndexFlat,0.01,6.6068,1.000
5,cosine,HNSW (ef=50),154.34,0.0512,0.999
